## **Installing Dependencies**

In [1]:
!pip install -qU --force-reinstall opentelemetry-api opentelemetry-sdk
!pip install -qU langchain-google-genai langgraph chromadb pypdf tiktoken langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk

## **Environment Setup**

In [2]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

## **Document Processing & Embeddings**
This code loads the PDF, breaks it into segments, and stores them in a ChromaDB vector database

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load your uploaded PDF
loader = PyPDFLoader("Customer-Service-Handbook.pdf")
docs = loader.load()

# 2. Chunking Strategy
# Using Recursive splitter to maintain paragraph integrity
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
splits = text_splitter.split_documents(docs)

# 3. Create Vector Store with updated Embedding Model
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# 4. Define the Retriever for the graph
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Successfully indexed {len(splits)} chunks into ChromaDB.")

Successfully indexed 32 chunks into ChromaDB.


## **The Graph Workflow (LangGraph + HITL)**
This is the core "brain" of the system. Gemini 2.5 Flash Lite will use its Thinking Budget to reason about whether it has enough information or if it needs a human.

In [5]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

# Define the State
class State(TypedDict):
    messages: Annotated[list, add_messages]
    needs_escalation: bool

# Initialize Gemini 2.5 Flash Lite with Thinking capability
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    thinking_budget=-1 # Allows dynamic reasoning to improve HITL accuracy
)

# Node 1: Processing
def assistant_node(state: State):
    query = state["messages"][-1].content
    docs = retriever.invoke(query)
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
    Context: {context}
    User Query: {query}

    Instruction: Answer the user based on the context.
    If the context is insufficient or the request is high-priority,
    reply ONLY with the word 'ESCALATE'.
    """
    response = llm.invoke(prompt)
    is_escalated = "ESCALATE" in response.content.upper()
    return {"messages": [response], "needs_escalation": is_escalated}

# Node 2: Human-in-the-Loop (Escalation)
def human_review_node(state: State):
    human_input = interrupt("ESCALATION TRIGGERED: Please provide the correct support answer:")
    return {"messages": [{"role": "assistant", "content": f"Verified Expert Response: {human_input}"}]}

# Build and Compile Graph
workflow = StateGraph(State)
workflow.add_node("assistant", assistant_node)
workflow.add_node("human_review", human_review_node)

workflow.add_edge(START, "assistant")
workflow.add_conditional_edges(
    "assistant",
    lambda x: "human_review" if x["needs_escalation"] else END
)
workflow.add_edge("human_review", END)

app = workflow.compile(checkpointer=MemorySaver())

# **Phase 1: The Initial Run**
In this code block, the graph will execute the `assistant_node`. If the AI decides it doesn't have enough info from your PDF, it will call `interrupt()`. This pauses the entire graph and "saves" its state to your `MemorySaver` checkpointer.

In [6]:
import uuid

# 1. Create a unique session ID
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# 2. Use a query that tests the PDF's specific policies
# Example: Something specific from the Arkansas Extension PDF
# What is the employee hoilday schedule?
query = {"messages": [{"role": "user", "content": "What is the specific timeline for returning a customer's phone call if the staff is in a meeting?"}]}

print("--- STARTING ASSISTANT ---")
# stream_mode="values" allows you to see the state updates as they happen
for event in app.stream(query, config, stream_mode="values"):
    if "messages" in event:
        last_msg = event["messages"][-1].content
        print(f"\nAssistant Status: {last_msg}")

print("\n--- RUN COMPLETE ---")

--- STARTING ASSISTANT ---

Assistant Status: What is the specific timeline for returning a customer's phone call if the staff is in a meeting?

Assistant Status: ESCALATE

Assistant Status: ESCALATE

--- RUN COMPLETE ---


# **Phase 2: Inspecting and Resuming**
If the Assistant responded with "ESCALATE", the graph is now "paused" at the `human_review` node. You need to check the state and provide the expert answer to resume.

In [7]:
from langgraph.types import Command

# 3. Check if the graph is waiting for a human
snapshot = app.get_state(config)

if snapshot.next:
    print(f"\n[!] SYSTEM PAUSED at node: {snapshot.next}")
    print("The AI has requested human intervention.")

    # 4. Provide the manual expert response
    # You can look up the answer in the Arkansas PDF
    human_expert_answer = input("Enter the expert response to provide to the user: ")

    # 5. RESUME the graph using Command(resume=...)
    print("\n--- RESUMING WITH EXPERT INPUT ---")
    for event in app.stream(Command(resume=human_expert_answer), config, stream_mode="values"):
        if "messages" in event:
            print(f"Final Verified Response: {event['messages'][-1].content}")
else:
    print("\n[✓] The Assistant answered successfully using the PDF context.")


[!] SYSTEM PAUSED at node: ('human_review',)
The AI has requested human intervention.
Enter the expert response to provide to the user: "All phone calls must be returned within 24 hours, even if the staff member is in meetings or away from their desk."

--- RESUMING WITH EXPERT INPUT ---
Final Verified Response: ESCALATE
Final Verified Response: Verified Expert Response: "All phone calls must be returned within 24 hours, even if the staff member is in meetings or away from their desk."
